In [1]:
import sys
import os

# Add the root project directory to sys.path
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

In [ ]:

import pandas as pd
# Load your cleaned data
df = pd.read_csv("../data/processed/cleaned_insurance_data.csv")

In [ ]:
from src.services.ab_testing import (
    test_gender_difference,
    test_province_difference,
    test_zipcode_margin_difference,
    test_zipcode_risk_difference
)

tests = [
    test_gender_difference(df),
    test_province_difference(df),
    test_zipcode_margin_difference(df),
    test_zipcode_risk_difference(df)
]

for result in tests:
    print(f"🧪 {result['test']}")
    print(f"Group A: {result['group_A']} vs Group B: {result['group_B']}")
    print(f"t-statistic: {result['t_statistic']:.4f}, p-value: {result['p_value']:.4e}")
    print(f"Reject null hypothesis? ➡️ {result['reject_null']}\n")


In [3]:
from src.services.ab_testing import (
    test_gender_difference,
    test_province_difference,
    test_zipcode_margin_difference,
    test_zipcode_risk_difference,
    province_claim_severity_anova,
    zipcode_margin_anova,
    zipcode_claim_frequency_chisq
)

# Run original t-tests
tests = [
    test_gender_difference(df),
    test_province_difference(df),
    test_zipcode_margin_difference(df),
    test_zipcode_risk_difference(df),
]

# Run extended tests
tests.extend([
    province_claim_severity_anova(df),
    zipcode_margin_anova(df),
    zipcode_claim_frequency_chisq(df)
])

# Print all results with proper formatting
for result in tests:
    print(f"🧪 {result['test']}")
    if "group_A" in result and "group_B" in result:
        print(f"Group A: {result['group_A']} vs Group B: {result['group_B']}")
    if "t_statistic" in result:
        print(f"t-statistic: {result['t_statistic']:.4f}, p-value: {result['p_value']:.4e}")
    else:
        print(f"p-value: {result['p_value']:.4e}")
    print(f"Reject null hypothesis? ➡️ {result['reject_null']}")
    if "num_groups" in result:
        print(f"Number of groups compared: {result['num_groups']}")
    print("-" * 60)


🧪 Gender vs Claim Frequency
Group A: Male vs Group B: Female
t-statistic: 0.5466, p-value: 5.8467e-01
Reject null hypothesis? ➡️ False
------------------------------------------------------------
🧪 Province vs Claim Severity
Group A: Gauteng vs Group B: KwaZulu-Natal
t-statistic: -3.2572, p-value: 1.1787e-03
Reject null hypothesis? ➡️ True
------------------------------------------------------------
🧪 Zipcode vs Margin
Group A: 2000 vs Group B: 122
t-statistic: 0.6394, p-value: 5.2259e-01
Reject null hypothesis? ➡️ False
------------------------------------------------------------
🧪 Zipcode vs Claim Frequency
Group A: 2000 vs Group B: 122
t-statistic: -2.9721, p-value: 2.9599e-03
Reject null hypothesis? ➡️ True
------------------------------------------------------------
🧪 ANOVA: Province vs Claim Severity
p-value: 1.0943e-05
Reject null hypothesis? ➡️ True
Number of groups compared: 9
------------------------------------------------------------
🧪 ANOVA: Zip Code vs Margin
p-value: 4.8



## ✅ Results Summary

| Hypothesis                                       | Metric          | Test Used  | Groups                   | p-value  | Decision             | Interpretation                                         |
| ------------------------------------------------ | --------------- | ---------- | ------------------------ | -------- | -------------------- | ------------------------------------------------------ |
| **H₀₁** No risk difference across provinces      | Claim Severity  | T-Test     | Gauteng vs KwaZulu-Natal | 0.0012   | ❌ Reject H₀         | Provinces differ significantly in avg. claim amount    |
| H₀₁ (Extended)                                   | Claim Severity  | ANOVA      | All 9 provinces          | 1.09e-05 | ❌ Reject H₀         | Strong evidence of regional claim severity differences |
| **H₀₂** No risk difference between zip codes     | Claim Frequency | T-Test     | Zip 2000 vs 122          | 0.0030   | ❌ Reject H₀         | Zip codes influence claim frequency                    |
| H₀₂ (Extended)                                   | Claim Frequency | Chi-square | Top 10 zip codes         | 8.69e-08 | ❌ Reject H₀         | Clear association between zip and claim behavior       |
| **H₀₃** No margin difference between zip codes   | Margin          | T-Test     | Zip 2000 vs 122          | 0.5226   | ✅ Fail to Reject H₀ | No margin difference found                             |
| H₀₃ (Extended)                                   | Margin          | ANOVA      | Top 10 zip codes         | 0.4842   | ✅ Fail to Reject H₀ | Margin remains stable across zip codes                 |
| **H₀₄** No risk difference between women and men | Claim Frequency | T-Test     | Male vs Female           | 0.5847   | ✅ Fail to Reject H₀ | No gender-based risk difference                        |

---


## 🔍 Detailed Analysis

### 🧪 H₀₁: Province vs Claim Severity

- Compared: **Gauteng** vs **KwaZulu-Natal**
- **p = 0.0012** → Statistically significant
- ✔ Claim amounts vary significantly by province
- **Implication:** Region should be a pricing and risk factor in underwriting models

---

### 🧪 H₀₂: Zip Code vs Claim Frequency

- Compared: Postal codes **2000** vs **122**
- **p = 0.0030** → Statistically significant
- ✔ Zip code location affects how often customers file claims
- **Implication:** Use geolocation to improve fraud detection or premium segmentation

---

### 🧪 H₀₃: Zip Code vs Margin

- Compared: Same top two zip codes
- **p = 0.5226** → Not statistically significant
- ✘ No difference in profit margin between these areas
- **Implication:** Premium calculations might already account for this risk

---

### 🧪 H₀₄: Gender vs Claim Frequency

- Compared: **Male** vs **Female**
- **p = 0.5847** → Not statistically significant
- ✘ Gender is not a meaningful predictor of claim behavior
- **Implication:** Avoid using gender in pricing or decision systems to maintain fairness

---

## 🔍 Interpretation

- ✅ **Provinces** influence **claim severity** → location is a key underwriting factor
- ✅ **Zip codes** significantly affect **claim frequency** → localized risk modeling is needed
- ❌ **Margin** does not vary significantly across top zip codes → pricing seems balanced
- ❌ **Gender** has no impact on claim likelihood → do not use gender in risk scoring

---

## 🧠 Business Recommendations

- 🎯 Incorporate **province** and **zipcode** into pricing and risk segmentation models
- 🚫 Avoid using **gender** in pricing — no evidence of predictive power
- 💡 Further explore margin stability — may reflect strong pricing policy
- 📈 Consider building **zip-level risk profiles** for fraud detection and personalized marketing

---